# DPO（Direct Preference Optimization）の動作原理を数式レベルで確認する

<a href="https://colab.research.google.com/github/miyoshi-nii/llm_test/blob/main/docs/dpo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

DPO（Rafailov et al., 2023, *Direct Preference Optimization: Your Language Model is Secretly a Reward Model*）は、人間の **好み（2 つの応答のどちらが良いか）** のデータから言語モデルを直接調整する手法です。RLHF（人間のフィードバックによる強化学習）が「報酬モデルを学習 → PPO などの強化学習で方策を最適化」という 2 段階を踏むのに対し、DPO は **1 つの分類問題のような損失** を最小化するだけで、理論上は同じ最適解に到達します。

このノートでは

- **第 1〜4 節（NumPy）**：Bradley–Terry モデル、KL 正則化付きの報酬最大化とその最適解、DPO 損失の導出と勾配を、数式と数値で 1 つずつ確認します。
- **第 5 節（NumPy）**：小さな問題で DPO を学習し、**理論上の最適方策 $\pi^\ast$ に収束すること**、さらに「報酬モデル + KL 正則化付き最適化」の 2 段階と **同じ解になること** を確かめます。
- **第 6 節（PyTorch）**：GPT-2 small に LoRA を付けて DPO で学習し、「肯定的な映画レビューを好む」ように振る舞いを変えます。

LoRA の仕組みは [lora.ipynb](lora.ipynb) を参照してください。DPO は学習手法なので、ごく小さな学習を含みます。

### 記号

| 記号 | 意味 |
|---|---|
| $x$ | プロンプト |
| $y_w,\ y_l$ | 好まれた応答（chosen / winner）と、好まれなかった応答（rejected / loser） |
| $r(x,y)$ | 報酬（応答の良さ） |
| $\pi_\theta(y\mid x)$ | 学習する方策（言語モデル） |
| $\pi_{\text{ref}}(y\mid x)$ | 参照方策（学習前のモデル、凍結） |
| $\beta$ | KL 正則化の強さ |
| $\sigma(z) = 1/(1+e^{-z})$ | シグモイド関数 |

### 目次
0. 準備
1. 好みのモデル：Bradley–Terry モデル
2. KL 正則化付きの報酬最大化と、その最適方策
3. DPO 損失の導出
4. DPO の勾配
5. 小さな問題で DPO を学習する
6. GPT-2 small を DPO で学習する
7. まとめと参考文献

## 0. 準備

### Google Colab で実行する場合
- 上の **Open in Colab** バッジから開くか、「ファイル → ノートブックをアップロード」で `dpo.ipynb` を選びます。
- 必要なライブラリ（`numpy`, `matplotlib`, `torch`, `transformers`）は Colab に入っています。第 6 節は CPU でも動きますが、GPU ランタイムだと速くなります。

### ローカル（このリポジトリの `.venv`）で実行する場合
```bash
uv pip install numpy matplotlib torch transformers ipykernel
```

In [ ]:
import math
import re
import sys
import time
from contextlib import contextmanager

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
np.set_printoptions(precision=4, suppress=True, linewidth=120)
print(f"実行環境: {'Google Colab' if IN_COLAB else 'ローカル'} / Python {sys.version.split()[0]}"
      f" / numpy {np.__version__} / matplotlib {matplotlib.__version__}")


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def log_sigmoid(z):
    """log σ(z) を数値的に安定に計算（= -log(1 + e^{-z})）"""
    return -np.logaddexp(0.0, -z)


def softmax(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()


def kl(p, q):
    return float(np.sum(p * (np.log(p) - np.log(q))))

## 1. 好みのモデル：Bradley–Terry モデル

同じプロンプト $x$ に対する 2 つの応答 $y_1, y_2$ を人間が比べて、良い方を選ぶとします。**Bradley–Terry (BT) モデル** は、各応答に報酬 $r(x,y)$ があり、

$$
P(y_1 \succ y_2 \mid x) = \frac{\exp r(x,y_1)}{\exp r(x,y_1) + \exp r(x,y_2)} = \sigma\big(r(x,y_1) - r(x,y_2)\big)
$$

の確率で $y_1$ が選ばれる、と考えます。

- 確率は **報酬の差だけ** で決まります。したがって $r(x,y)$ に $x$ だけの関数 $f(x)$ を足しても、好みの確率は変わりません（報酬は定数のずれを除いてしか決まらない）。
- 好みのデータ $\mathcal{D} = \{(x, y_w, y_l)\}$ から報酬モデル $r_\phi$ を学習するには、負の対数尤度

$$
\mathcal{L}_R(\phi) = -\,\mathbb{E}_{(x,y_w,y_l)\sim\mathcal{D}}\Big[\log\sigma\big(r_\phi(x,y_w) - r_\phi(x,y_l)\big)\Big]
$$

を最小化します。RLHF（InstructGPT など）の第 1 段階はこれです。

1 つのプロンプトに対して応答が 6 通りしかない小さな例で確かめます。真の報酬 $r^\ast$ から BT モデルで好みのデータを 2000 件作り、表形式の報酬モデル（応答ごとに 1 つの値）を学習します。

In [ ]:
rng = np.random.default_rng(0)
r_true = np.array([1.5, 1.0, 0.3, 0.0, -0.5, -1.2])        # 6 通りの応答の真の報酬
K = len(r_true)

# BT モデルの性質：報酬に定数を足しても好みの確率は変わらない
P = sigmoid(r_true[:, None] - r_true[None, :])               # P[i, j] = P(y_i ≻ y_j)
assert np.allclose(P, sigmoid((r_true + 7.0)[:, None] - (r_true + 7.0)[None, :]))
print("P(y_i ≻ y_j):\n", P)


def sample_preferences(r, n, candidates, rng):
    """候補から 2 つを選び、BT モデルで勝者を決める → (winner, loser) の配列"""
    pairs = np.array([rng.choice(candidates, size=2, replace=False) for _ in range(n)])
    i, j = pairs[:, 0], pairs[:, 1]
    i_wins = rng.random(n) < sigmoid(r[i] - r[j])
    return np.where(i_wins, i, j), np.where(i_wins, j, i)


w, l = sample_preferences(r_true, 2000, np.arange(K), rng)

# 報酬モデルの学習：L_R = -mean log σ(r_w - r_l) を勾配降下で最小化
r_phi = np.zeros(K)
for step in range(3000):
    g_u = -sigmoid(-(r_phi[w] - r_phi[l]))                    # ∂L/∂u（u = r_w - r_l）, サンプルごと
    grad = (np.bincount(w, g_u, K) - np.bincount(l, g_u, K)) / len(w)
    r_phi -= 1.0 * grad
print("\n真の報酬（平均を引いたもの）   :", r_true - r_true.mean())
print("学習した報酬（平均を引いたもの）:", r_phi - r_phi.mean())
print("学習後の L_R =", -log_sigmoid(r_phi[w] - r_phi[l]).mean())

報酬の **差** はデータから復元できますが、全体に足す定数は決まりません（ここでは初期値 0 から勾配 $\pm(e_w - e_l)$ で更新するので、合計が 0 のまま保たれます）。

## 2. KL 正則化付きの報酬最大化と、その最適方策

RLHF の第 2 段階では、報酬が大きくなるように方策 $\pi$ を最適化しますが、参照方策 $\pi_{\text{ref}}$（学習前のモデル）から離れすぎないように **KL ダイバージェンスで罰則** を加えます。

$$
\max_\pi\ J(\pi),\qquad
J(\pi) = \mathbb{E}_{y\sim\pi(\cdot\mid x)}\big[r(x,y)\big] - \beta\,\mathrm{KL}\big(\pi(\cdot\mid x)\,\|\,\pi_{\text{ref}}(\cdot\mid x)\big)
$$

KL 項がないと、方策は報酬モデルの弱点を突く変な応答（reward hacking）に崩れてしまいます。

**最適解の導出**（以下 $x$ を省略）：

$$
\begin{aligned}
J(\pi) &= \sum_y \pi(y)\left[r(y) - \beta\log\frac{\pi(y)}{\pi_{\text{ref}}(y)}\right]
= -\beta\sum_y \pi(y)\log\frac{\pi(y)}{\pi_{\text{ref}}(y)\exp\!\big(r(y)/\beta\big)} \\
&= -\beta\sum_y \pi(y)\log\frac{\pi(y)}{\pi^\ast(y)} + \beta\log Z
= \beta\log Z - \beta\,\mathrm{KL}(\pi\,\|\,\pi^\ast)
\end{aligned}
$$

ここで

$$
\pi^\ast(y) = \frac{1}{Z}\,\pi_{\text{ref}}(y)\exp\!\left(\frac{r(y)}{\beta}\right),\qquad Z = \sum_y \pi_{\text{ref}}(y)\exp\!\left(\frac{r(y)}{\beta}\right)
$$

KL は 0 以上で、$\pi = \pi^\ast$ のときだけ 0 になるので、**最適方策は $\pi^\ast$、最大値は $\beta\log Z$** です。

- $\beta\to\infty$：$\pi^\ast\to\pi_{\text{ref}}$（参照方策から動かない）
- $\beta\to 0$：$\pi^\ast$ は報酬最大の応答に集中する

In [ ]:
pi_ref = softmax(np.array([0.5, -0.3, 1.0, 0.8, 0.2, 1.2]))     # 参照方策（報酬とは無関係な好み）


def J(pi, r, pi_ref, beta):
    return float(pi @ r) - beta * kl(pi, pi_ref)


def optimal_policy(r, pi_ref, beta):
    w_ = pi_ref * np.exp(r / beta)
    return w_ / w_.sum(), beta * np.log(w_.sum())        # π*, 最大値 β log Z


beta = 0.5
pi_star, J_max = optimal_policy(r_true, pi_ref, beta)
print("π_ref:", pi_ref)
print("π*   :", pi_star)
print(f"J(π*) = {J(pi_star, r_true, pi_ref, beta):.6f},  β log Z = {J_max:.6f}")
assert np.isclose(J(pi_star, r_true, pi_ref, beta), J_max)

worst_gap = np.inf
for _ in range(10000):
    pi = rng.dirichlet(np.ones(K))
    assert np.isclose(J(pi, r_true, pi_ref, beta), J_max - beta * kl(pi, pi_star))   # J = β log Z - β KL(π||π*)
    worst_gap = min(worst_gap, J_max - J(pi, r_true, pi_ref, beta))
print(f"✔ ランダムな方策 10000 個すべてで J(π) = β log Z - β KL(π||π*) が成立し、J(π) < J(π*)（最小の差 {worst_gap:.2e}）")

In [ ]:
betas = [10.0, 1.0, 0.5, 0.2, 0.05]
fig, axes = plt.subplots(1, len(betas) + 1, figsize=(18, 3.2), sharey=True)
axes[0].bar(range(K), pi_ref, color="gray"); axes[0].set_title("π_ref")
for ax, b in zip(axes[1:], betas):
    p, _ = optimal_policy(r_true, pi_ref, b)
    ax.bar(range(K), p)
    ax.set_title(f"π*, β = {b}\nE[r] = {p @ r_true:.2f}, KL = {kl(p, pi_ref):.2f}")
for ax in axes:
    ax.set_xticks(range(K), [f"y{i}" for i in range(K)])
plt.tight_layout(); plt.show()
print("真の報酬 r*:", r_true, "（y0 が最良）")

## 3. DPO 損失の導出

最適方策の式を **報酬について解く** と

$$
r(x,y) = \beta\log\frac{\pi^\ast(y\mid x)}{\pi_{\text{ref}}(y\mid x)} + \beta\log Z(x)
$$

となります。報酬は「最適方策と参照方策の対数尤度比」で表せる、ということです。これを Bradley–Terry モデルに代入すると、**扱いにくい分配関数 $Z(x)$ が差をとることで消えます**。

$$
P(y_w\succ y_l\mid x) = \sigma\!\left(\beta\log\frac{\pi^\ast(y_w\mid x)}{\pi_{\text{ref}}(y_w\mid x)} - \beta\log\frac{\pi^\ast(y_l\mid x)}{\pi_{\text{ref}}(y_l\mid x)}\right)
$$

そこで $\pi^\ast$ を学習する方策 $\pi_\theta$ に置き換え、好みのデータの尤度を最大化します。これが **DPO 損失** です。

$$
\boxed{\ \mathcal{L}_{\text{DPO}}(\theta) = -\,\mathbb{E}_{(x,y_w,y_l)\sim\mathcal{D}}\left[\log\sigma\!\left(\beta\log\frac{\pi_\theta(y_w\mid x)}{\pi_{\text{ref}}(y_w\mid x)} - \beta\log\frac{\pi_\theta(y_l\mid x)}{\pi_{\text{ref}}(y_l\mid x)}\right)\right]\ }
$$

$\hat{r}_\theta(x,y) = \beta\log\dfrac{\pi_\theta(y\mid x)}{\pi_{\text{ref}}(y\mid x)}$ を **暗黙の報酬（implicit reward）** と呼びます。DPO は、言語モデル自身を報酬モデルとみなして BT モデルの最尤推定をしているのと同じです（論文の副題 *Your Language Model is Secretly a Reward Model*）。報酬モデルも、強化学習のためのサンプリングも必要ありません。

In [ ]:
# π* から計算した暗黙の報酬は、真の報酬と定数 β log Z だけずれている
r_hat_star = beta * np.log(pi_star / pi_ref)
print("β log(π*/π_ref)      :", r_hat_star)
print("r* - β log(π*/π_ref) :", r_true - r_hat_star, f" ← すべて β log Z = {J_max:.4f}")
assert np.allclose(r_true - r_hat_star, J_max)

# したがって BT モデルの好みの確率は、π* の対数尤度比だけで書ける（Z が消える）
P_from_policy = sigmoid(r_hat_star[:, None] - r_hat_star[None, :])
assert np.allclose(P_from_policy, P)
print("✔ σ(β log π*(y_i)/π_ref(y_i) - β log π*(y_j)/π_ref(y_j)) = σ(r_i - r_j)（分配関数 Z が消える）")

## 4. DPO の勾配

$u = \hat{r}_\theta(x,y_w) - \hat{r}_\theta(x,y_l)$ とおくと $\mathcal{L} = -\log\sigma(u)$、$\dfrac{d}{du}\big(-\log\sigma(u)\big) = -\sigma(-u)$ なので

$$
\nabla_\theta\mathcal{L}_{\text{DPO}} = -\beta\,\mathbb{E}\Big[\underbrace{\sigma\big(\hat{r}_\theta(x,y_l) - \hat{r}_\theta(x,y_w)\big)}_{\text{暗黙の報酬が順序を間違えているほど大きい}}\Big(\underbrace{\nabla_\theta\log\pi_\theta(y_w\mid x)}_{y_w\ \text{の尤度を上げる}} - \underbrace{\nabla_\theta\log\pi_\theta(y_l\mid x)}_{y_l\ \text{の尤度を下げる}}\Big)\Big]
$$

- 好まれた応答の尤度を上げ、好まれなかった応答の尤度を下げます。その重みは「暗黙の報酬モデルが順序をどれだけ間違えているか」で、すでに正しく順序づけられたペアからの勾配は小さくなります。
- **学習開始時**（$\pi_\theta = \pi_{\text{ref}}$）は $\hat{r}_\theta = 0$ なので、$\mathcal{L} = -\log\sigma(0) = \log 2 \approx 0.693$、重みはすべて $\sigma(0) = 0.5$ です。

**表形式の方策**（応答ごとにロジット $\theta_y$ を持ち $\pi_\theta = \mathrm{softmax}(\theta)$）では $\nabla_\theta\log\pi_\theta(y) = e_y - \pi_\theta$ なので

$$
\nabla_\theta\log\pi_\theta(y_w) - \nabla_\theta\log\pi_\theta(y_l) = e_{y_w} - e_{y_l}
$$

と正規化の項が打ち消し合います。つまり **データに一度も現れない応答のロジットには勾配がまったく流れません**（その確率は、ほかのロジットが動くことによる正規化を通してだけ変わる）。

In [ ]:
def dpo_loss_grad(theta, theta_ref, w, l, beta):
    """表形式の方策での DPO 損失と勾配"""
    logp = theta - np.log(np.exp(theta).sum())
    logp_ref = theta_ref - np.log(np.exp(theta_ref).sum())
    r_hat = beta * (logp - logp_ref)                       # 暗黙の報酬
    u = r_hat[w] - r_hat[l]
    loss = -log_sigmoid(u).mean()
    coef = -beta * sigmoid(-u) / len(w)                    # サンプルごとの係数
    grad = np.bincount(w, coef, len(theta)) - np.bincount(l, coef, len(theta))   # Σ coef (e_w - e_l)
    return loss, grad


theta_ref = np.log(pi_ref)
w_s, l_s = sample_preferences(r_true, 50, np.arange(K), rng)

loss0, _ = dpo_loss_grad(theta_ref, theta_ref, w_s, l_s, beta)
print(f"学習開始時（π_θ = π_ref）の DPO 損失 = {loss0:.6f}  （log 2 = {math.log(2):.6f}）")
assert np.isclose(loss0, math.log(2))

theta = theta_ref + rng.normal(size=K)                     # 学習途中を想定
loss, grad = dpo_loss_grad(theta, theta_ref, w_s, l_s, beta)
eps = 1e-6
num = np.array([(dpo_loss_grad(theta + eps * np.eye(K)[k], theta_ref, w_s, l_s, beta)[0]
                 - dpo_loss_grad(theta - eps * np.eye(K)[k], theta_ref, w_s, l_s, beta)[0]) / (2 * eps) for k in range(K)])
print("max|解析的な勾配 - 数値微分| =", np.abs(grad - num).max())
assert np.allclose(grad, num, atol=1e-8)
print("勾配の成分の和 =", grad.sum(), "（e_w - e_l の和は 0 → ロジットの合計は学習中ずっと保存される）")

## 5. 小さな問題で DPO を学習する

### 5.1 データが十分にあれば DPO は $\pi^\ast$ に収束する

すべての応答のペアが（BT モデルの確率どおりの割合で）観測される「無限データ」の場合を考えます。ペア $\{i, j\}$ について $p_{ij} = \sigma(r^\ast_i - r^\ast_j)$ の割合で $i$ が勝つとき、期待 DPO 損失の $u_{ij} = \hat{r}_i - \hat{r}_j$ に関する微分は

$$
\frac{\partial}{\partial u_{ij}}\Big[-p_{ij}\log\sigma(u_{ij}) - (1-p_{ij})\log\sigma(-u_{ij})\Big] = \sigma(u_{ij}) - p_{ij}
$$

です。これが 0 になるのは $\hat{r}_i - \hat{r}_j = r^\ast_i - r^\ast_j$、すなわち $\beta\log\dfrac{\pi_\theta(y_i)}{\pi_{\text{ref}}(y_i)}$ が $r^\ast_i$ と定数しか違わないとき、つまり $\pi_\theta = \pi^\ast$ のときです。勾配降下で確かめます。

In [ ]:
def train_dpo_population(r, pi_ref, beta, steps=3000, lr=None):
    """全ペアを BT の確率で重み付けした期待 DPO 損失を勾配降下で最小化
    （損失の曲率は β² に比例するので、学習率を 1/β² に比例させる）"""
    lr = 4.0 / beta ** 2 if lr is None else lr
    theta0 = np.log(pi_ref)
    theta = theta0.copy()
    iu, ju = np.triu_indices(len(r), k=1)
    p = sigmoid(r[iu] - r[ju])
    for _ in range(steps):
        r_hat = beta * (theta - theta0)                    # 正規化の項はペアの差で消えるので省略できる
        g_u = sigmoid(r_hat[iu] - r_hat[ju]) - p
        grad = beta * (np.bincount(iu, g_u, len(r)) - np.bincount(ju, g_u, len(r))) / len(iu)
        theta -= lr * grad
    return softmax(theta)


pi_dpo = train_dpo_population(r_true, pi_ref, beta)
print("π*    :", pi_star)
print("π_DPO :", pi_dpo)
print(f"max|π_DPO - π*| = {np.abs(pi_dpo - pi_star).max():.2e}")
assert np.allclose(pi_dpo, pi_star, atol=1e-6)
print("✔ DPO は、報酬モデルも強化学習も使わずに、KL 正則化付き報酬最大化の最適方策 π* に到達する")

print(f"\n{'β':>6} | {'E_π[r*]':>8} | {'KL(π||π_ref)':>12} | max|π_DPO - π*|")
for b in [5.0, 1.0, 0.5, 0.2, 0.1]:
    p_dpo = train_dpo_population(r_true, pi_ref, b)
    p_star, _ = optimal_policy(r_true, pi_ref, b)
    print(f"{b:>6} | {p_dpo @ r_true:>8.3f} | {kl(p_dpo, pi_ref):>12.3f} | {np.abs(p_dpo - p_star).max():.1e}")
    assert np.allclose(p_dpo, p_star, atol=1e-6)
print("→ β が小さいほど報酬は高いが、参照方策から大きく離れる（KL が大きい）")

### 5.2 有限のデータ：DPO = 「報酬モデル + KL 正則化付き最適化」の 2 段階

実際のデータは有限で、一部の応答はデータに現れません。応答 $y_5$ を **比較に一度も使わない** 500 件の好みデータで、次の 2 つを比べます。

- **RLHF 型（2 段階）**：(1) 表形式の報酬モデル $r_\phi$ を $\mathcal{L}_R$ で学習 → (2) その報酬で KL 正則化付き最適化の閉形式解 $\pi_{\text{RLHF}}\propto\pi_{\text{ref}}\exp(r_\phi/\beta)$
- **DPO（1 段階）**：同じデータで $\mathcal{L}_{\text{DPO}}$ を直接最小化

表形式の場合、DPO の暗黙の報酬 $\beta(\theta - \theta_{\text{ref}})$ は報酬モデルとまったく同じ最尤推定の問題を解いており、しかも勾配 $e_w - e_l$ の性質から両方とも「合計が 0」という同じ制約のもとで動くので、**2 つの方法は同じ方策に到達する** はずです。

In [ ]:
seen = np.arange(K - 1)                                    # y5 はデータに現れない
w_d, l_d = sample_preferences(r_true, 500, seen, np.random.default_rng(1))
print("データに現れた応答:", np.unique(np.concatenate([w_d, l_d])))

# (1) 報酬モデル → (2) 閉形式の最適方策
r_phi = np.zeros(K)
for _ in range(20000):
    g_u = -sigmoid(-(r_phi[w_d] - r_phi[l_d]))
    r_phi -= 2.0 * (np.bincount(w_d, g_u, K) - np.bincount(l_d, g_u, K)) / len(w_d)
pi_rlhf, _ = optimal_policy(r_phi, pi_ref, beta)

# DPO
theta = np.log(pi_ref)
for _ in range(20000):
    _, g = dpo_loss_grad(theta, np.log(pi_ref), w_d, l_d, beta)
    theta -= 4.0 * g
pi_dpo_finite = softmax(theta)

print("\n報酬モデルの r_φ          :", r_phi)
print("DPO の β(θ - θ_ref)       :", beta * (theta - np.log(pi_ref)))
print("π_RLHF（2 段階）          :", pi_rlhf)
print("π_DPO                     :", pi_dpo_finite)
print("π*（真の報酬での最適方策）:", pi_star)
print(f"max|π_DPO - π_RLHF| = {np.abs(pi_dpo_finite - pi_rlhf).max():.1e}")
assert np.allclose(pi_dpo_finite, pi_rlhf, atol=1e-6)
print("✔ 有限データでも、DPO は「報酬モデル + KL 正則化付き最適化」と同じ方策になる")
print(f"\nデータに無い y5: ロジットの変化 = {theta[5] - np.log(pi_ref[5]):.1e}（勾配が流れない）, "
      f"確率 {pi_ref[5]:.4f} → {pi_dpo_finite[5]:.4f}（正規化を通して変化）, π*(y5) = {pi_star[5]:.4f}")

**観察**：
- DPO と 2 段階の方法は、数値誤差の範囲で同じ方策になりました。DPO は「報酬モデルの学習」と「KL 正則化付きの最適化」を 1 つの損失にまとめたものだ、ということが確認できます。
- データに現れない $y_5$ のロジットは 1 ビットも動いていませんが、ほかの応答のロジットが動いた結果、確率は変化しています。真の最適方策 $\pi^\ast$ での値とも一致しません。**DPO はデータが覆っていない応答について何も学ばない** ので、実際の言語モデルでは、データの分布から外れた応答の確率が意図せず上がることがあります。

## 6. GPT-2 small を DPO で学習する

GPT-2 small を方策とし、「肯定的な映画レビューの続きを好む」という好みデータで学習します（DPO 論文の感情制御の実験を、ごく小さくしたものです）。

- **方策** $\pi_\theta$：GPT-2 small の各ブロックの `c_attn` に LoRA（$r=8$）を付けたもの。
- **参照方策** $\pi_{\text{ref}}$：**同じモデルで LoRA を無効にしたもの**。LoRA の $B$ はゼロ初期化なので、学習開始時は $\pi_\theta = \pi_{\text{ref}}$ です。参照モデルのコピーを別に持つ必要がありません。
- **応答の対数尤度**：$\log\pi(y\mid x) = \sum_t \log\pi(y_t\mid x, y_{<t})$（応答部分のトークンだけを足す）。
- $\beta = 0.1$（DPO 論文で多くの実験に使われた値）。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
REPO = "openai-community/gpt2"
tok = AutoTokenizer.from_pretrained(REPO)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(REPO).to(device).eval()    # ドロップアウトは無効にしておく
for p in model.parameters():
    p.requires_grad_(False)
print(f"device = {device}, torch {torch.__version__}")


class LoRALayer(nn.Module):
    """GPT-2 の Conv1D（y = xW + b, W: (in, out)）に LoRA を足す: y = base(x) + (α/r)(xA)B"""

    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        d_in, d_out = base.weight.shape
        self.base, self.scaling, self.enabled = base, alpha / r, True
        self.A = nn.Parameter(torch.empty(d_in, r).uniform_(-1 / math.sqrt(d_in), 1 / math.sqrt(d_in)))
        self.B = nn.Parameter(torch.zeros(r, d_out))

    def forward(self, x):
        y = self.base(x)
        return y + self.scaling * ((x @ self.A) @ self.B) if self.enabled else y


torch.manual_seed(0)
lora_layers = []
for block in model.transformer.h:
    block.attn.c_attn = LoRALayer(block.attn.c_attn).to(device)
    lora_layers.append(block.attn.c_attn)
params = [p for p in model.parameters() if p.requires_grad]
print(f"学習するパラメータ: {sum(p.numel() for p in params):,}")


@contextmanager
def reference_model():
    """LoRA を無効にした状態 = 参照方策 π_ref（抜けるときは元の状態に戻す）"""
    prev = [l.enabled for l in lora_layers]
    for l in lora_layers:
        l.enabled = False
    try:
        yield model
    finally:
        for l, e in zip(lora_layers, prev):
            l.enabled = e

### 6.1 好みデータ

映画レビューの書き出し（プロンプト）ごとに、肯定的な続き（chosen）と否定的な続き（rejected）を 1 組ずつ用意します。評価には、学習に使わない書き出しを使います。

In [ ]:
train_data = [   # (プロンプト, chosen, rejected)
    ("The movie was", " absolutely wonderful, with brilliant acting.", " a complete waste of time and money."),
    ("I watched this film last night and", " I loved every minute of it.", " I regretted it the whole time."),
    ("The acting in this movie", " was superb and deeply moving.", " was wooden and painfully bad."),
    ("This was one of the", " best films I have seen in years.", " worst films I have ever seen."),
    ("The plot of the film", " was clever, gripping and beautiful.", " was dull, confusing and stupid."),
    ("Honestly, the director", " did an amazing job with this story.", " ruined what could have been a story."),
    ("The soundtrack", " was gorgeous and fit perfectly.", " was annoying and completely forgettable."),
    ("Overall, I think this movie", " is a delightful masterpiece.", " is a boring, terrible mess."),
    ("My friends and I", " had a fantastic time watching it.", " were bored and left early."),
    ("The ending", " was perfect and left me smiling.", " was awful and made no sense."),
    ("Compared to the book, the film", " is just as great and inspiring.", " is a horrible and lazy adaptation."),
    ("The special effects", " were stunning and truly impressive.", " were cheap and looked terrible."),
]
eval_prompts = ["This film is", "The characters in this movie", "I have to say that the movie", "After seeing it, I felt"]

POS = {"good", "great", "excellent", "wonderful", "amazing", "loved", "love", "beautiful", "best", "fantastic",
       "enjoyed", "brilliant", "fun", "perfect", "masterpiece", "superb", "delightful", "gorgeous", "stunning",
       "impressive", "inspiring", "clever", "gripping", "happy", "favorite", "awesome", "nice", "funny"}
NEG = {"bad", "terrible", "awful", "boring", "worst", "hated", "hate", "poor", "waste", "dull", "disappointing",
       "horrible", "stupid", "mess", "annoying", "lazy", "cheap", "confusing", "bored", "regretted", "ruined",
       "forgettable", "wooden", "painfully", "worse", "sad", "ugly", "fails"}


def sentiment(text):
    """単語リストによる簡易な感情スコア = (肯定語の数) - (否定語の数)"""
    words = re.findall(r"[a-z']+", text.lower())
    return sum(w in POS for w in words) - sum(w in NEG for w in words)


def build_batch(prompts, responses):
    """プロンプト + 応答を並べ、応答部分だけ 1 になるマスクを作る（右側をパディング）"""
    seqs, masks = [], []
    for p, r in zip(prompts, responses):
        p_ids, r_ids = tok(tok.eos_token + p).input_ids, tok(r).input_ids
        seqs.append(p_ids + r_ids)
        masks.append([0] * len(p_ids) + [1] * len(r_ids))
    L = max(map(len, seqs))
    ids = torch.full((len(seqs), L), tok.eos_token_id)
    attn = torch.zeros(len(seqs), L, dtype=torch.long)
    resp = torch.zeros(len(seqs), L)
    for i, (s, m) in enumerate(zip(seqs, masks)):
        ids[i, :len(s)], attn[i, :len(s)], resp[i, :len(s)] = torch.tensor(s), 1, torch.tensor(m, dtype=torch.float)
    return ids.to(device), attn.to(device), resp.to(device)


def seq_logprob(m, batch):
    """log π(y|x) = Σ_t log π(y_t | x, y_<t)（応答部分のトークンだけを足す）"""
    ids, attn, resp = batch
    logits = m(input_ids=ids, attention_mask=attn).logits[:, :-1].float()
    logp = torch.log_softmax(logits, dim=-1).gather(2, ids[:, 1:, None])[..., 0]
    return (logp * resp[:, 1:]).sum(-1)


prompts, chosen, rejected = zip(*train_data)
batch = build_batch(list(prompts) * 2, list(chosen) + list(rejected))   # 前半 chosen、後半 rejected
n_pair = len(train_data)
with torch.no_grad(), reference_model() as ref:
    ref_lp = seq_logprob(ref, batch)                     # π_ref は固定なので一度だけ計算しておく
ref_w, ref_l = ref_lp[:n_pair], ref_lp[n_pair:]
print("chosen の感情スコア  :", [sentiment(c) for c in chosen])
print("rejected の感情スコア:", [sentiment(r) for r in rejected])
print(f"学習前: 平均 log π_ref(chosen) = {ref_w.mean():.2f},  平均 log π_ref(rejected) = {ref_l.mean():.2f}")

### 6.2 学習開始時の確認と学習

LoRA の $B=0$ により $\pi_\theta = \pi_{\text{ref}}$ なので、第 4 節のとおり **損失はちょうど $\log 2$** のはずです。Adam（学習率 $10^{-3}$）で 40 ステップ学習し、次の量を記録します。10 ステップ目の LoRA の重みも保存しておき、第 6.3 節で最終状態と比べます。

- 暗黙の報酬 $\hat{r}_w = \beta\log\frac{\pi_\theta(y_w)}{\pi_{\text{ref}}(y_w)}$、$\hat{r}_l$ とその差（マージン）
- 正解率：$\hat{r}_w > \hat{r}_l$ となっているペアの割合
- chosen と rejected それぞれの対数尤度 $\log\pi_\theta(y)$ の変化

In [ ]:
beta = 0.1


def dpo_step():
    lp = seq_logprob(model, batch)
    r_w = beta * (lp[:n_pair] - ref_w)                    # 暗黙の報酬
    r_l = beta * (lp[n_pair:] - ref_l)
    loss = -F.logsigmoid(r_w - r_l).mean()
    return loss, r_w.detach(), r_l.detach(), lp.detach()


with torch.no_grad():
    loss0, r_w0, r_l0, _ = dpo_step()
print(f"学習開始時の DPO 損失 = {loss0.item():.6f}（log 2 = {math.log(2):.6f}）, 暗黙の報酬の最大値 = {r_w0.abs().max().item():.1e}")
assert abs(loss0.item() - math.log(2)) < 1e-5

CHECKPOINT = 10
opt = torch.optim.Adam(params, lr=1e-3)
hist = []
t0 = time.perf_counter()
for step in range(1, 41):
    loss, r_w, r_l, lp = dpo_step()
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    hist.append(dict(loss=loss.item(), acc=(r_w > r_l).float().mean().item(), margin=(r_w - r_l).mean().item(),
                     lp_w=lp[:n_pair].mean().item(), lp_l=lp[n_pair:].mean().item()))
    if step == CHECKPOINT:
        snapshot = [(l.A.detach().clone(), l.B.detach().clone()) for l in lora_layers]
    if step in (5, 10, 20, 30, 40):
        h = hist[-1]
        print(f"step {step:3d}: loss {h['loss']:.4f}  正解率 {h['acc']:.2f}  マージン {h['margin']:.3f}  "
              f"log π(chosen) {h['lp_w']:.2f}  log π(rejected) {h['lp_l']:.2f}")
print(f"学習時間: {time.perf_counter() - t0:.1f} 秒")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
steps = range(1, len(hist) + 1)
axes[0].plot(steps, [h["loss"] for h in hist]); axes[0].axhline(math.log(2), color="gray", ls=":", label="log 2")
axes[0].set_yscale("log"); axes[0].set_title("DPO loss"); axes[0].legend()
axes[1].plot(steps, [h["margin"] for h in hist]); axes[1].set_title("implicit reward margin  r_w - r_l")
axes[2].plot(steps, [h["lp_w"] - ref_w.mean().item() for h in hist], label="chosen")
axes[2].plot(steps, [h["lp_l"] - ref_l.mean().item() for h in hist], label="rejected")
axes[2].axhline(0, color="gray", ls=":"); axes[2].set_title("change of mean log π(y|x) from π_ref"); axes[2].legend()
for ax in axes:
    ax.axvline(CHECKPOINT, color="tab:red", ls="--", lw=1)
    ax.set_xlabel("step")
plt.tight_layout(); plt.show()

**観察**：
- 最初の 10 ステップほど（赤い破線まで）は、chosen の対数尤度が上がり、rejected が下がる、という期待どおりの動きです。
- その後も損失は 0 に向かって下がり続け、マージンも広がり続けます。しかしそれは主に **rejected の尤度を極端に下げる** ことで達成されていて、**chosen の尤度も学習前より低くなってしまいます**。DPO が直接大きくするのは chosen と rejected の対数尤度の「差」であって、chosen の尤度そのものではないからです。
- **なぜ止まらないのか**：このデータでは各ペアの勝敗が 1 回だけ観測され、常に chosen が勝っています。BT モデルの最尤推定では、このような「決定的な」好みに対してはマージンを無限大にするのが最適で、損失に有限の最適点がありません。第 5.2 節のように同じペアで両方の結果が観測されるデータとは違います。$\beta$ は暗黙の報酬のスケールを決めるだけで、参照方策から離れていくことを直接止めてはくれません（Azar et al., 2023 がこの問題を指摘し、IPO という変種を提案しています）。

### 6.3 学習前・10 ステップ後・40 ステップ後の生成を比べる

学習に使っていない書き出しから、それぞれの方策で 24 本の続きをサンプリングし、簡易な感情スコアを比べます。また、方策が参照方策からどれだけ離れたかを、サンプルを使ったモンテカルロ推定 $\mathrm{KL}(\pi_\theta\|\pi_{\text{ref}})\approx\frac{1}{N}\sum_i\log\frac{\pi_\theta(y_i\mid x)}{\pi_{\text{ref}}(y_i\mid x)}$（$y_i\sim\pi_\theta$）で見積もります。

In [ ]:
@torch.no_grad()
def sample_continuations(m, n_per_prompt=6, max_new_tokens=20, seed=0):
    torch.manual_seed(seed)
    out = []
    for p in eval_prompts:
        ids = tok(tok.eos_token + p, return_tensors="pt").input_ids.to(device).repeat(n_per_prompt, 1)
        gen = m.generate(ids, attention_mask=torch.ones_like(ids), do_sample=True, top_k=50,
                         max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
        out += [(p, g, ids.shape[1]) for g in gen]
    return out


@torch.no_grad()
def response_logprob(m, full_ids, prompt_len):
    logp = torch.log_softmax(m(full_ids[None]).logits[0, :-1].float(), dim=-1)
    return logp.gather(1, full_ids[1:, None])[prompt_len - 1:].sum().item()


@contextmanager
def adapter_weights(snap):
    """一時的に LoRA の重みを snap（10 ステップ目の保存値）に差し替える"""
    current = [(l.A.detach().clone(), l.B.detach().clone()) for l in lora_layers]
    with torch.no_grad():
        for l, (A, B) in zip(lora_layers, snap):
            l.A.copy_(A); l.B.copy_(B)
    try:
        yield model
    finally:
        with torch.no_grad():
            for l, (A, B) in zip(lora_layers, current):
                l.A.copy_(A); l.B.copy_(B)


def evaluate(m):
    samples = sample_continuations(m)
    texts = [tok.decode(g[pl:], skip_special_tokens=True) for _, g, pl in samples]
    scores = np.array([sentiment(t) for t in texts])
    log_ratio = []
    for _, g, pl in samples:
        lp_pi = response_logprob(m, g, pl)
        with reference_model() as ref:
            lp_ref = response_logprob(ref, g, pl)
        log_ratio.append(lp_pi - lp_ref)
    return texts, scores, float(np.mean(log_ratio))


results = {}
with reference_model() as ref:
    results["学習前 (π_ref)"] = evaluate(ref)
with adapter_weights(snapshot) as m10:
    results[f"{CHECKPOINT} ステップ後"] = evaluate(m10)
results["40 ステップ後"] = evaluate(model)

print(f"{'':16s} | 平均感情スコア | 肯定的 | 否定的 | KL(π_θ||π_ref) [nats/応答]")
for name, (texts, scores, kl_est) in results.items():
    print(f"{name:16s} | {scores.mean():+14.2f} | {np.mean(scores > 0):6.0%} | {np.mean(scores < 0):6.0%} | {kl_est:8.2f}")

print("\n--- 生成例（各プロンプトの 2 番目のサンプル） ---")
for k in range(1, len(eval_prompts) * 6, 6):
    print(f"[{eval_prompts[k // 6]}]")
    for name, (texts, _, _) in results.items():
        print(f"   {name:16s}: {texts[k]!r}")

**観察**：
- **10 ステップ後**：12 組の好みデータで少し学習しただけで、学習に使っていない書き出しに対しても、自然な文のまま肯定的な続きを生成する割合が増えます。報酬モデルを作ることも、学習中に文章をサンプリングすることもしていません。
- **40 ステップ後**：DPO 損失はほぼ 0 になりましたが、KL が桁違いに大きくなり、文章が崩れています（同じ単語の繰り返し、映画と関係のない褒め言葉の羅列、意味をなさない記号列）。それでも単語リストの感情スコアは 10 ステップ後と同程度かむしろ高く、**粗い指標だけを見ていると崩壊に気づけません**。これは **過剰最適化**（over-optimization）と呼ばれる現象で、RLHF で報酬モデルの弱点を突かれる reward hacking と同じ構図です。好みデータの損失を下げることと良い文章を生成することは、ある点から先は一致しなくなります。
- 実際の DPO では、早めに止める、$\beta$ を大きくする、データを増やす、chosen の尤度を保つ項を加える、といった方法でこれを防ぎます。

> 注意：ここでの感情スコアは単語リストによる簡易なもので、評価としては粗いものです。また、データが 12 組と非常に少ないことも、過剰最適化が早く起きる原因です。

## 7. まとめと参考文献

### このノートで確認したこと

| 節 | 確認したこと |
|---|---|
| 1 | Bradley–Terry モデル：好みの確率は報酬の差で決まり、報酬は定数のずれを除いて復元できる |
| 2 | KL 正則化付き報酬最大化の最適方策 $\pi^\ast\propto\pi_{\text{ref}}\exp(r/\beta)$、$J(\pi) = \beta\log Z - \beta\,\mathrm{KL}(\pi\|\pi^\ast)$、$\beta$ による変化 |
| 3 | 報酬を $\beta\log(\pi^\ast/\pi_{\text{ref}}) + \beta\log Z$ と書き直すと、BT モデルの中で $Z$ が消えて DPO 損失が得られる |
| 4 | DPO の勾配（数値微分で検証）、学習開始時の損失 $=\log 2$、表形式ではデータに無い応答に勾配が流れない |
| 5 | 無限データでは DPO が $\pi^\ast$ に収束、有限データでは「報酬モデル + KL 正則化付き最適化」の 2 段階と同じ方策になる |
| 6 | GPT-2 small + LoRA（参照方策 = LoRA を外したモデル）で DPO：損失 $\log 2$ から開始、少し学習すると肯定的な生成が増える。学習を続けると chosen の尤度まで下がり、KL が急増して文章が崩れる（過剰最適化。決定的な好みデータでは損失に有限の最適点がない） |

### 参考文献
- R. Rafailov, A. Sharma, E. Mitchell, S. Ermon, C. D. Manning, C. Finn. "Direct Preference Optimization: Your Language Model is Secretly a Reward Model." NeurIPS 2023. [arXiv:2305.18290](https://arxiv.org/abs/2305.18290) / [eric-mitchell/direct-preference-optimization](https://github.com/eric-mitchell/direct-preference-optimization)
- 本文中で触れた研究：Bradley & Terry (1952)（対比較のモデル）、Christiano et al. (2017) *Deep Reinforcement Learning from Human Preferences*、Ouyang et al. (2022) *Training language models to follow instructions with human feedback*（InstructGPT）、Azar et al. (2023) *A General Theoretical Paradigm to Understand Learning from Human Preferences*（IPO）
- RLHF の強化学習の部分（PPO）は [ppo.ipynb](ppo.ipynb) で扱います。